# Project 3: Analyzing 2023 SIPA Alumni Employment Data
### Welcome to my third and final project!

If you're considering graduate school in international affairs, one of the biggest questions on your mind is probably: "Will I actually get a job?" And if you're an international student, you might be even more worried: "Is it going to be way harder for me?" As someone who's attending grad school as an international student, I decided to dig into Columbia's School of International and Public Affairs (SIPA) employment data to answer these questions.

- **Dataset(s) to be used:** [[link](https://www.sipa.columbia.edu/pathways-careers/employment-statistics)]
  - The dataset comprises 638 observations with 97 variables covering employment status, industry sectors, geographic locations, and demographic characteristics. 
  - The dataset won't be uploaded to Github for data privacy considerations.
- **Analysis question:** 
  - Do international students really struggle more to find jobs than domestic students? How will they choose the job sector?
- **Columns that will (likely) be used:**
  - ['International']
  - ['Outcome']
  - ['Sector']
- **Hypothesis**: 
  - International students will find it harder to find jobs than domestic students, and will prefer private sectors for salary considerations.

In [35]:
import pandas as pd
import plotly.express as px
from IPython.display import HTML
# ensure the visualizations render properly across Vscode, Jupyter Book, etc.
# https://plotly.com/python/renderers/

In [36]:
file_path = r'D:\Python\CIC\Project 3\2023 Outcome Survey Data_Aidan.csv'

try:
    df = pd.read_csv(file_path, skiprows=2)
except FileNotFoundError:
    print("Data file not found. Skipping load to avoid errors.")
    df = None

In [37]:
# Won't show the head of the dataframe for privacy considerations
# df.head()

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 642 entries, 0 to 641
Data columns (total 97 columns):
 #   Column                                                                                    Non-Null Count  Dtype  
---  ------                                                                                    --------------  -----  
 0   International                                                                             638 non-null    object 
 1   Filled out survey                                                                         638 non-null    object 
 2   APSIA Job Category                                                                        575 non-null    object 
 3   APSIA Sector Category                                                                     372 non-null    object 
 4   Date Reported - NEW                                                                       216 non-null    object 
 5   Outcome                                                  

### Step 1: Cleaning up mostly-empty columns

Before diving into analysis, I need to deal with a common problem in survey data: columns where hardly anyone actually filled in an answer. Remember that the dataset has 97 columns, but not all of them are actually useful. Some columns might only have 5 or 10 responses out of 638 students. That's basically useless for analysis.

**Why 300?** That's roughly 47% of the 638 total students. I'm saying "if less than half the people bothered to answer this question, it's probably not reliable enough to analyze."

**The result:** Instead of 97 columns (many of which are nearly empty), I now have a cleaner dataset with only the columns that have substantial data.

In [39]:
df = df.loc[:, df.count() > 300]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 642 entries, 0 to 641
Data columns (total 28 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   International                      638 non-null    object
 1   Filled out survey                  638 non-null    object
 2   APSIA Job Category                 575 non-null    object
 3   APSIA Sector Category              372 non-null    object
 4   Outcome                            638 non-null    object
 5   Concentration                      638 non-null    object
 6   Degree                             638 non-null    object
 7   Sector                             468 non-null    object
 8   Detailed Industry                  474 non-null    object
 9   Organization Name                  474 non-null    object
 10  Job Title                          474 non-null    object
 11  Job City                           431 non-null    object
 12  Job Stat

Missing values are addressed through explicit categorization as "Unknown" rather than deletion, preserving sample size while maintaining data integrity. This approach allows for transparent handling of incomplete records in subsequent analyses.

In [40]:
df['International'] = df['International'].fillna('Unknown')
df['Outcome'] = df['Outcome'].fillna('Unknown')

### Step 2: Simple data overview

In [41]:
# Total sample size
print(f"Total sample size: {len(df)}\n")

# International student distribution
print("International Student Distribution:")
intl_dist = pd.DataFrame({
    'Status': df['International'].value_counts().index,
    'Count': df['International'].value_counts().values,
    'Percentage': (df['International'].value_counts().values / len(df) * 100).round(1)
})
intl_dist['Percentage'] = intl_dist['Percentage'].astype(str) + '%'
display(intl_dist)

# Employment outcome distribution
print("\nEmployment Outcome Distribution:")
outcome_dist = pd.DataFrame({
    'Outcome': df['Outcome'].value_counts().index,
    'Count': df['Outcome'].value_counts().values,
    'Percentage': (df['Outcome'].value_counts().values / len(df) * 100).round(1)
})
outcome_dist['Percentage'] = outcome_dist['Percentage'].astype(str) + '%'
display(outcome_dist)

# OSA nationality distribution (top 15)
print("\nTop 15 Countries of Citizenship:")
osa_dist = pd.DataFrame({
    'Country': df['OSA Country of Citizenship Lookup'].value_counts().head(15).index,
    'Count': df['OSA Country of Citizenship Lookup'].value_counts().head(15).values,
    'Percentage': (df['OSA Country of Citizenship Lookup'].value_counts().head(15).values / len(df) * 100).round(1)
})
osa_dist['Percentage'] = osa_dist['Percentage'].astype(str) + '%'
display(osa_dist)

Total sample size: 642

International Student Distribution:


,Status,Count,Percentage
0,Yes,372,57.9%
1,No,266,41.4%
2,Unknown,4,0.6%



Employment Outcome Distribution:


,Outcome,Count,Percentage
0,Job,468,72.9%
1,Unreported,118,18.4%
2,Still Seeking Employment,41,6.4%
3,Pursuing Further Education,6,0.9%
4,Not seeking / Other Intention,5,0.8%
5,Unknown,4,0.6%



Top 15 Countries of Citizenship:


,Country,Count,Percentage
0,UNITED STATES,251,39.1%
1,CHINA,104,16.2%
2,INDIA,56,8.7%
3,JAPAN,27,4.2%
4,INDONESIA,20,3.1%
5,MEXICO,13,2.0%
6,COLOMBIA,11,1.7%
7,SINGAPORE,9,1.4%
8,SOUTH KOREA,9,1.4%
9,BRAZIL,8,1.2%


The first question I wanted to answer was straightforward: do international and domestic students have different employment outcomes? But before jumping into comparisons, I needed to understand what "employment outcome" actually means in this dataset.

Looking at the outcome distribution, I found five main categories:
- **Job** (468 students, 73%) - Successfully employed
- **Unreported** (118 students, 18%) - No information provided
- **Still Seeking Employment** (41 students, 6%) - Actively job hunting
- **Pursuing Further Education** (6 students, 1%) - Continuing studies
- **Not seeking / Other Intention** (5 students, 1%) - Not in the job market

### Step 3: Creating a Fair Comparison

To compare international and domestic students fairly, I created a stacked bar chart showing the percentage breakdown of outcomes for each group. 

In [42]:
# Clean Outcome Category Mapping 
df["Outcome_Clean"] = df["Outcome"].replace({
    "Job": "Job",
    "Still Seeking Employment": "Still Seeking",
    "Unreported": "Unreported",
    "Pursuing Further Education": "Further Education",
    "Not seeking / Other Intention": "Other Intention",
    "Unknown": "Unknown"
})

# Remove International = "Unknown"
df_clean = df[df["International"].isin(["Yes", "No"])]

# Map Yes/No to readable labels
df_clean["Status"] = df_clean["International"].map({"Yes": "International", "No": "Domestic"})

# ===== Count outcomes per group =====
outcome_counts = df_clean.groupby(["Status", "Outcome_Clean"]).size().reset_index(name="Count")

# Convert to percentage within each group
total_per_status = outcome_counts.groupby("Status")["Count"].transform("sum")
outcome_counts["Percentage"] = outcome_counts["Count"] / total_per_status * 100

# ===== Visualization: 100% stacked bar chart =====
fig = px.bar(
    outcome_counts,
    x="Status",
    y="Percentage",
    color="Outcome_Clean",
    title="Employment Outcome Distribution of International vs Domestic Students (%)",
    text=outcome_counts["Percentage"].map(lambda x: f"{x:.1f}%"),
    width=600
)

fig.update_layout(
    barmode="stack",
    yaxis_title="Percentage (%)",
    yaxis_range=[0, 100]
)

fig.update_traces(textposition="inside")

# Wherever you have fig.show() replace it with this code:
HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

C:\Users\25025\AppData\Local\Temp\ipykernel_53680\2620020524.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



**What the First Chart Reveals**

At first glance, domestic students appear to have a slight employment advantage (76.7% vs 71.0%). However, the high "unreported" rate for international students (22.3% vs 13.2%) raises questions. Are these students:
- Actually employed but didn't respond to the survey?
- Working in their home countries (outside the tracking system)?
- Facing more barriers and therefore less likely to report?

### Step 4: Digging deeper - Employment by country of citizenship

The "international student" label is quite broad. To understand these nuances, I broke down employment outcomes by specific countries and filtered to only include countries with more than 5 students in the dataset because small samples can be misleading. This filter kept 16 countries in the analysis, with sample sizes ranging from 6 (Pakistan) to 251 (United States).

In [43]:
# Clean Outcome Category Mapping 
df["Outcome_Clean"] = df["Outcome"].replace({
    "Job": "Job",
    "Still Seeking Employment": "Still Seeking",
    "Unreported": "Unreported",
    "Pursuing Further Education": "Further Education",
    "Not seeking / Other Intention": "Other Intention",
    "Unknown": "Unknown"
})

# Filter out unknown countries 
df_clean3 = df[df["OSA Country of Citizenship Lookup"].notna()]

# Count total samples per country 
country_counts = df_clean3.groupby("OSA Country of Citizenship Lookup").size().reset_index(name="Total_Count")

# Keep only countries with sample size > 5 
valid_countries = country_counts[country_counts["Total_Count"] > 5]["OSA Country of Citizenship Lookup"]
df_filtered = df_clean3[df_clean3["OSA Country of Citizenship Lookup"].isin(valid_countries)]

# Group by Country and Outcome
outcome_counts = df_filtered.groupby(["OSA Country of Citizenship Lookup", "Outcome_Clean"]).size().reset_index(name="Count")

# Convert counts to percentage within each country 
total_per_country = outcome_counts.groupby("OSA Country of Citizenship Lookup")["Count"].transform("sum")
outcome_counts["Percentage"] = outcome_counts["Count"] / total_per_country * 100

# Merge total counts for annotation 
outcome_counts = outcome_counts.merge(country_counts, on="OSA Country of Citizenship Lookup")

fig2 = px.bar(
    outcome_counts,
    x="OSA Country of Citizenship Lookup",
    y="Percentage",
    color="Outcome_Clean",
    title="Employment Outcome Distribution by Country of Citizenship (Sample > 5)",
    text=outcome_counts["Percentage"].map(lambda x: f"{x:.1f}%"),
    width=1000
)

fig2.update_layout(
    barmode="stack",
    yaxis_title="Percentage (%)",
    yaxis_range=[0, 110],  
    xaxis_title="Country of Citizenship",
    bargap=0.2
)

# Show percentages inside bars
fig2.update_traces(textposition="inside")

# Add total sample size as annotations above bars
for country in outcome_counts["OSA Country of Citizenship Lookup"].unique():
    total = outcome_counts[outcome_counts["OSA Country of Citizenship Lookup"] == country]["Total_Count"].iloc[0]
    fig2.add_annotation(
        x=country,
        y=105,  
        text=f"N={total}",
        showarrow=False,
        font=dict(size=12, color="black"),
        xanchor='center'
    )

# Wherever you have fig.show() replace it with this code:
HTML(fig2.to_html(include_plotlyjs="cdn", full_html=False))

The country-specific breakdown reveals significant variation that the simple "international vs domestic" comparison completely misses:

**High Employment Rate Countries (>85%):**
- Colombia (90.9%), Mexico (92.3%), Singapore (100%)
- These students appear to face fewer barriers, possibly due to strong institutional partnerships or regional economic factors

**Moderate Employment Rate Countries (70-85%):**
- United States (76.9%), India (82.1%), Brazil (87.5%)
- Represents the "typical" outcome for most students

**Lower Employment Rate Countries (<60%):**
- Japan (40.7%), Saudi Arabia (57.1%), China (55.8%)

**The Japan and China Patterns Are Striking:**

Japan shows a particularly unusual pattern: 59.3% unreported and only 40.7% employed among those who reported. This could indicate:
- Many students returning to Japan for employment (outside the tracking system)
- Cultural factors affecting survey response rates
- Genuine employment challenges requiring further investigation

China, with 104 students (the largest international cohort), shows 55.8% employment but 35.6% unreported. Given the sample size, this unreported rate is concerning and suggests data collection challenges with this population.

**An Important Caveat**

The sample sizes above each bar (annotated as "N=X") remind us that even countries with >5 students still have relatively small samples. For instance, Singapore's perfect 100% employment rate looks impressive, but it's based on only 9 students - hardly enough to conclude that all SIPA graduates from Singapore will find jobs easily.

### Step 5: Sector Preferences - Testing the "Private Sector" Hypothesis

My initial hypothesis was that international students would cluster in private sector jobs for salary and visa sponsorship reasons. The reality turned out to be more nuanced.

Domestic students are far more likely to work in nonprofits (37% vs 19%). This makes sense when you consider:
1. **Visa limitations** - Nonprofits may be less willing/able to sponsor work visas
2. **Salary constraints** - Nonprofits typically pay less, which is harder to justify for visa sponsorship
3. **Mission alignment** - International students may prioritize sectors that offer clearer paths to long-term US employment

**Government sector similarity** (24% vs 26%) is somewhat surprising given that many government jobs require US citizenship. This suggests that:
- Some international students work for international organizations (UN, World Bank) classified as "Public"
- Some return to their home countries for government work
- A subset may have work authorization allowing government employment

In [44]:
# Filter for records with valid sector information and group by international status and sector
sector_by_status = df[df['Sector'].notna()].groupby(['International', 'Sector']).size().reset_index(name='Count')

# Create readable status labels
sector_by_status['Status'] = sector_by_status['International'].map({'Yes': 'International', 'No': 'Domestic'})

# Calculate percentage within each student status group
# This normalizes the data so percentages sum to 100% for each status separately
sector_by_status['Percentage'] = sector_by_status.groupby('Status')['Count'].transform(lambda x: x / x.sum() * 100)

# Create grouped bar chart comparing sector distribution
fig3 = px.bar(
    sector_by_status[sector_by_status['Status'].isin(['International', 'Domestic'])], 
    x='Sector', 
    y='Percentage',
    color='Status',
    barmode='group',
    title='International vs Domestic Students - Industry Distribution Comparison (%)',
    labels={'Percentage': 'Percentage (%)', 'Sector': 'Industry Sector'}
)

# Wherever you have fig.show() replace it with this code:
HTML(fig3.to_html(include_plotlyjs="cdn", full_html=False))

## Conclusions and Limitations

### Main Findings

1. **Employment rates are similar but not identical:** Domestic students show slightly higher employment rates (76.7% vs 71.0%), but the difference is modest

2. **Country of origin matters significantly:** The "international student" category masks huge variation - from 100% employment (Singapore) to 40.7% (Japan)

3. **The nonprofit gap is real:** Domestic students are twice as likely to work in nonprofits, likely due to visa and compensation constraints

4. **Private sector isn't the international student haven I expected:** Employment is split fairly evenly across sectors

### Data Quality Concerns

The high "unreported" rate, especially for international students (22.3%), is a significant limitation. This could mean:
- Actual employment rates are higher than reported
- Students working internationally aren't captured
- Survey fatigue or privacy concerns affect response rates